Mystery 1:
There is a customer in the database that has gone completely silent. Find the customers who have placed the fewest orders and generated the least revenue.

Section 1: Pull all customers and their total order count

Section 2: Join with order details to calculate total revenue per customer

Section 3: Apply ranking functions to sort by least activity

Section 4: Single CTE query identifying the bottom 5 customers by revenue and order count


In [ ]:
--Section 1:
SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalOrders ASC
--Section 2:
SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalRevenue ASC
--Section 3: 
SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerId, c.CustomerCompanyName
HAVING COUNT(o.OrderId) < 5
ORDER BY TotalRevenue ASC
--Section 4:
;WITH CustomerActivity AS (
    SELECT 
        c.CustomerId,
        c.CustomerCompanyName,
        COUNT(o.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue
    FROM Sales.Customer c
    JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY c.CustomerId, c.CustomerCompanyName
)
SELECT TOP 5
    CustomerId,
    CustomerCompanyName,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue
FROM CustomerActivity
ORDER BY TotalRevenue ASC

(Command completed successfully)

Explination: We need to investigate inactive customers by first countering the orders per customer. We then add the revenue and filter to the low order counts, wrapping it up by using a CTE to get the bottom 5 customers. By running the query we can see that customer VMLOG is the most forgotten customer with only 2 orders and $100.80 in revenue.

Mystery 2:
One of our shippers has barely moved any packages. Find the shipper with the lowest involvment in orders and determine if they are pulling their weight.

Section 1: Pull all shippers and count their total orders

Section 2: Join with order details to calculate total revenue handled per shipper

Section 3: Calculate percentage of total orders each shipper handles

Section 4: Single CTE identifying the least active shipper and their share of business

In [ ]:
--Section 1:
SELECT 
    s.ShipperId,
    s.ShipperCompanyName,
    COUNT(o.OrderId) AS TotalOrders
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
GROUP BY s.ShipperId, s.ShipperCompanyName
ORDER BY TotalOrders ASC
--Section 2: 
SELECT 
    s.ShipperId,
    s.ShipperCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY s.ShipperId, s.ShipperCompanyName
ORDER BY TotalRevenue ASC
--Section 3:
SELECT 
    s.ShipperCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    ROUND(COUNT(o.OrderId) * 100.0 / (SELECT COUNT(*) FROM Sales.[Order]), 2) AS OrderSharePct
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY s.ShipperId, s.ShipperCompanyName
ORDER BY OrderSharePct ASC
--Section 4:
;WITH ShipperActivity AS (
    SELECT 
        s.ShipperCompanyName,
        COUNT(o.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue,
        ROUND(COUNT(o.OrderId) * 100.0 / (SELECT COUNT(*) FROM Sales.[Order]), 2) AS OrderSharePct
    FROM Sales.Shipper s
    JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY s.ShipperId, s.ShipperCompanyName
)
SELECT 
    ShipperCompanyName,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue,
    OrderSharePct
FROM ShipperActivity
ORDER BY TotalOrders ASC

Explination: We can get the shippers performance by first counting the orders per shipper and then adding the revenue. This leads to calculating the share of total orders for each one and combining it all into one last CTE. From the query it shows that shipper ZHISN is the most silent shipper with the lowest order count at 645 orders and only $407,750.82 in revenue.Also they only handle 77.71% of their expected share.

Mystery 3:
We have customers who are complaining about late deliveries but we dont know who is responsible. Find which shippers are consistently taking the longest to deliver and if it varies by country.

Section 1: Pull all orders and calculate delivery time using OrderDate and ShippedDate

Section 2: Join with shipper data and group average delivery time by shipper

Section 3: Break delivery times down further by destination country

Section 4: Single CTE identifying the worst performing shipper per country using window functions

In [ ]:
--Section 1:
SELECT 
    o.OrderId,
    o.ShipperId,
    DATEDIFF(DAY, o.OrderDate, o.ShipToDate) AS DeliveryDays
FROM Sales.[Order] o
WHERE o.ShipToDate IS NOT NULL
ORDER BY DeliveryDays DESC
--Section 2:
SELECT 
    s.ShipperCompanyName,
    AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS AvgDeliveryDays
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
WHERE o.ShipToDate IS NOT NULL
GROUP BY s.ShipperCompanyName
ORDER BY AvgDeliveryDays DESC
--Section 3:
SELECT 
    s.ShipperCompanyName,
    o.ShipToCountry,
    AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS AvgDeliveryDays
FROM Sales.Shipper s
JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
WHERE o.ShipToDate IS NOT NULL
GROUP BY s.ShipperCompanyName, o.ShipToCountry
ORDER BY AvgDeliveryDays DESC
--Section 4:
;WITH ShippingPerformance AS (
    SELECT 
        s.ShipperCompanyName,
        o.ShipToCountry,
        AVG(DATEDIFF(DAY, o.OrderDate, o.ShipToDate)) AS AvgDeliveryDays,
        COUNT(o.OrderId) AS TotalOrders
    FROM Sales.Shipper s
    JOIN Sales.[Order] o ON s.ShipperId = o.ShipperId
    WHERE o.ShipToDate IS NOT NULL
    GROUP BY s.ShipperCompanyName, o.ShipToCountry
)
SELECT TOP 10
    ShipperCompanyName,
    ShipToCountry,
    AvgDeliveryDays,
    TotalOrders
FROM ShippingPerformance
ORDER BY AvgDeliveryDays DESC

Explination: I started with calculating delivery days per order and then averaging them by shipper. I moved on to breaking it down by the country and finalizing it into a CTE. Shipper GVSUA was thr slowest delivery shipper taking an average of 14 days to deliever to Belgium. Shipper ETYNR appeared 5 times in the top 10 and they were slower going to Switzerland and Austria being between 12 and 13 days.

Mystery 4:
Our boss wants to know who their top performers are but also who might be giving away too much in discounts. Find the employees using the highest value orders and mark any with suspicious discount patterns.

Section 1: Pull all orders and link them to employees

Section 2: Calculate total revenue and average discount per employee

Section 3: Rank employees by total revenue generated

Section 4: Single CTE identifying top employees by revenue while flagging those with above average discount rates

In [ ]:
--Section 1:
SELECT 
    e.EmployeeId,
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    COUNT(o.OrderId) AS TotalOrders
FROM HumanResources.Employee e
JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName
ORDER BY TotalOrders DESC
--Section 2:
SELECT 
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM HumanResources.Employee e
JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName
ORDER BY TotalRevenue DESC
--Section 3:
SELECT 
    e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    AVG(od.DiscountPercentage) AS AvgDiscount
FROM HumanResources.Employee e
JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName
ORDER BY TotalRevenue DESC
--Section 4:
;WITH EmployeePerformance AS (
    SELECT 
        e.EmployeeFirstName + ' ' + e.EmployeeLastName AS EmployeeName,
        COUNT(o.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue,
        AVG(od.DiscountPercentage) AS AvgDiscount
    FROM HumanResources.Employee e
    JOIN Sales.[Order] o ON e.EmployeeId = o.EmployeeId
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY e.EmployeeId, e.EmployeeFirstName, e.EmployeeLastName
)
SELECT 
    EmployeeName,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue,
    ROUND(AvgDiscount * 100, 2) AS AvgDiscountPct,
    CASE 
        WHEN AvgDiscount > 0.06 THEN 'Flag - High Discounter'
        ELSE 'OK'
    END AS DiscountFlag
FROM EmployeePerformance
ORDER BY TotalRevenue DESC

Explination: We need to be able to see employee performance so we first count the orders per employee and then add the total revenue. Then we averaged the discount rates and finished it with a CTE that shows high discounters. Yael Peled is our best employee with the highest revenue at $250,187.45 and 420 orders but, their marked as a high discounter at 6.13%. We can also see that Russell King had a discount pattern at 7.36% but, they rank 5th in revenue meaning they are probably hurting their contribution.

Mystery 5:
Something strange is happening with our sales patterns. There are certain months that are way above or below average and nobody knows why. Expose which months stand out compared to the yearly average.

Section 1: Pull total revenue grouped by month

Section 2: Calculate the overall yearly average monthly revenue

Section 3: Compare each month against the average and calculate variance

Section 4: Single CTE using window functions to flag months that deviate significantly from the average and rank them

In [ ]:
--Section 1:
SELECT 
    MONTH(o.OrderDate) AS MonthNumber,
    DATENAME(MONTH, o.OrderDate) AS MonthName,
    SUM(od.LineAmount) AS TotalRevenue
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY MONTH(o.OrderDate), DATENAME(MONTH, o.OrderDate)
ORDER BY MonthNumber ASC
--Section 2:
SELECT 
    AVG(MonthlyRevenue) AS AvgMonthlyRevenue
FROM (
    SELECT 
        MONTH(o.OrderDate) AS MonthNumber,
        SUM(od.LineAmount) AS MonthlyRevenue
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY MONTH(o.OrderDate)
) AS MonthlyTotals
--Section 3:
SELECT 
    DATENAME(MONTH, o.OrderDate) AS MonthName,
    SUM(od.LineAmount) AS TotalRevenue,
    SUM(od.LineAmount) - AVG(SUM(od.LineAmount)) OVER () AS VarianceFromAvg
FROM Sales.[Order] o
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY MONTH(o.OrderDate), DATENAME(MONTH, o.OrderDate)
ORDER BY VarianceFromAvg DESC
--Section 4:
;WITH MonthlyRevenue AS (
    SELECT 
        DATENAME(MONTH, o.OrderDate) AS MonthName,
        MONTH(o.OrderDate) AS MonthNumber,
        SUM(od.LineAmount) AS TotalRevenue,
        SUM(od.LineAmount) - AVG(SUM(od.LineAmount)) OVER () AS VarianceFromAvg
    FROM Sales.[Order] o
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY MONTH(o.OrderDate), DATENAME(MONTH, o.OrderDate)
)
SELECT 
    MonthName,
    ROUND(TotalRevenue, 2) AS TotalRevenue,
    ROUND(VarianceFromAvg, 2) AS VarianceFromAvg,
    CASE
        WHEN VarianceFromAvg > 0 THEN 'Above Average'
        ELSE 'Below Average'
    END AS Status
FROM MonthlyRevenue
ORDER BY VarianceFromAvg DESC

Explination: We want to see the seasonal sales patterns as we noticed anamolies when seasons change. We can see this by grouping revenue by the months and then calculating the total of our average, $112,871.55. Then we compare that average to the months and end it with a CTE that shows the above and below average of each month. April is our highest paid month at $190,329.95, which is $77,458.40 above our average. June is our worst performing month at $39,088.00, which is $73,783.55 below average. This means we should push more sales around June to bring it up to average. 

Mystery 6:
Products are sitting in the catalog but nobody is ordering them. Find the products with the lowest quantity ever ordered and determine if they should be removed.

Section 1: Pull all products and their total quantity ordered

Section 2: Join with category data to see which categories they belong to

Section 3: Calculate each product's share of total orders

Section 4: Single CTE identifying bottom products by quantity with category context

In [ ]:
--Section 1:
SELECT 
    p.ProductId,
    p.ProductName,
    SUM(od.Quantity) AS TotalQuantityOrdered
FROM Production.Product p
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY p.ProductId, p.ProductName
ORDER BY TotalQuantityOrdered ASC
--Section 2:
SELECT 
    p.ProductName,
    c.CategoryName,
    SUM(od.Quantity) AS TotalQuantityOrdered
FROM Production.Product p
JOIN Production.Category c ON p.CategoryId = c.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY p.ProductName, c.CategoryName
ORDER BY TotalQuantityOrdered ASC
--Section 3:
SELECT 
    p.ProductName,
    c.CategoryName,
    SUM(od.Quantity) AS TotalQuantityOrdered,
    SUM(od.LineAmount) AS TotalRevenue
FROM Production.Product p
JOIN Production.Category c ON p.CategoryId = c.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY p.ProductName, c.CategoryName
ORDER BY TotalQuantityOrdered ASC
-- Section 4:
;WITH ProductActivity AS (
    SELECT 
        p.ProductName,
        c.CategoryName,
        SUM(od.Quantity) AS TotalQuantityOrdered,
        SUM(od.LineAmount) AS TotalRevenue
    FROM Production.Product p
    JOIN Production.Category c ON p.CategoryId = c.CategoryId
    JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
    GROUP BY p.ProductName, c.CategoryName
)
SELECT TOP 5
    ProductName,
    CategoryName,
    TotalQuantityOrdered,
    ROUND(TotalRevenue, 2) AS TotalRevenue
FROM ProductActivity
ORDER BY TotalQuantityOrdered ASC

Explination: Here we are able to see low performing products by first counting the quantity that is ordered per product and adding catergory info, then revenue. We then put it into a final CTE to see the bottom 5. Product AOZBW in the Meat/Poultry category is the most understocked with only 95 units ever ordered. Then next to it is Product KSZOI in the Condiments category with only 122 units.

Mystery 7:
An employee in our supplier list is barely contributing to our catalog. Find suppliers providing the fewest products and not them as they are at risk.

Section 1: Pull all suppliers and count how many products they provide

Section 2: Join with order data to see how often their products are ordered

Section 3: Calculate total revenue generated per supplier

Section 4: Single CTE identifying underperforming suppliers by product count and revenue

In [ ]:
--Section 1:
SELECT 
    s.SupplierId,
    s.SupplierCompanyName,
    COUNT(p.ProductId) AS TotalProducts
FROM Production.Supplier s
JOIN Production.Product p ON s.SupplierId = p.SupplierId
GROUP BY s.SupplierId, s.SupplierCompanyName
ORDER BY TotalProducts ASC
--Section 2:
SELECT 
    s.SupplierCompanyName,
    COUNT(DISTINCT p.ProductId) AS TotalProducts,
    COUNT(od.OrderId) AS TotalOrders
FROM Production.Supplier s
JOIN Production.Product p ON s.SupplierId = p.SupplierId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY s.SupplierId, s.SupplierCompanyName
ORDER BY TotalOrders ASC
--Section 3:
SELECT 
    s.SupplierCompanyName,
    COUNT(DISTINCT p.ProductId) AS TotalProducts,
    COUNT(od.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Production.Supplier s
JOIN Production.Product p ON s.SupplierId = p.SupplierId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY s.SupplierId, s.SupplierCompanyName
ORDER BY TotalRevenue ASC
--Section 4:
;WITH SupplierActivity AS (
    SELECT 
        s.SupplierCompanyName,
        COUNT(DISTINCT p.ProductId) AS TotalProducts,
        COUNT(od.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue
    FROM Production.Supplier s
    JOIN Production.Product p ON s.SupplierId = p.SupplierId
    JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
    GROUP BY s.SupplierId, s.SupplierCompanyName
)
SELECT TOP 5
    SupplierCompanyName,
    TotalProducts,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue
FROM SupplierActivity
ORDER BY TotalRevenue ASC


Explination: Similar to the last one we counted suppliers products then added order volume, then revenue and finally used a CTE to identify the bottom 5 so we can see the most underperforming suppliers. From the output we see that Supplier UNAHG is underperforming the most at only 1 product, 51 orders and only $4,782.60 in revenue. Supplier ZRYDZ also only carries 1 product generating $6,664.75. These two Suppliers are the most at risk for their jobs. 

Mystery 8:
Revenue is leaking somewhere and someone is giving out heavy discounts and its costing us. Find which categories are losing the most money through discounting.

Section 1: Pull all order details and calculate revenue loss per order using discount percentage

Section 2: Join with product and category data to group losses by category

Section 3: Calculate average discount rate per category and rank them

Section 4: Single CTE combining revenue loss, average discount, and order count to identify the worst offending category

In [ ]:
--Section 1:
SELECT 
    od.OrderId,
    od.ProductId,
    od.DiscountPercentage,
    od.LineAmount - od.DiscountedLineAmount AS RevenueLoss
FROM Sales.OrderDetail od
ORDER BY RevenueLoss DESC
--Section 2:
SELECT 
    c.CategoryName,
    SUM(od.LineAmount - od.DiscountedLineAmount) AS TotalRevenueLoss
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY TotalRevenueLoss DESC
--Section 3:
SELECT 
    c.CategoryName,
    AVG(od.DiscountPercentage) AS AvgDiscount,
    SUM(od.LineAmount - od.DiscountedLineAmount) AS TotalRevenueLoss,
    COUNT(od.OrderId) AS TotalOrders
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY TotalRevenueLoss DESC
--Section 4:
;WITH DiscountActivity AS (
    SELECT 
        c.CategoryName,
        AVG(od.DiscountPercentage) AS AvgDiscount,
        SUM(od.LineAmount - od.DiscountedLineAmount) AS TotalRevenueLoss,
        COUNT(od.OrderId) AS TotalOrders
    FROM Production.Category c
    JOIN Production.Product p ON c.CategoryId = p.CategoryId
    JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
    GROUP BY c.CategoryName
)
SELECT 
    CategoryName,
    ROUND(AvgDiscount * 100, 2) AS AvgDiscountPct,
    ROUND(TotalRevenueLoss, 2) AS TotalRevenueLoss,
    TotalOrders
FROM DiscountActivity
ORDER BY TotalRevenueLoss DESC

Explination: I was able to see the discounted revenue loss by calculating loss per order in section 1, then grouped by the catgeories and added the average discount rates. Finally we wraped it up with the final CTE so we can see the discounted revenue loss. Beverages from the chart is the highest revenue loss at $18,658.77 and a 6.19% average discount rate. Meat/Poultry has the highest discount rate at 6.45% but fewer orders and Grains/Cereals is the our most efficient category with the lowest loss at $4,982.21.

 Mystery 9:
Some categories look busy on paper but arent making real money. Find categories with high order volume but low revenue .

Section 1: Pull total orders and revenue per category

Section 2: Calculate revenue per order for each category

Section 3: Rank categories by order volume and separately by revenue

Section 4: Single CTE comparing both rankings to flag categories where order rank is much higher than revenue rank

In [ ]:
--Section 1:
SELECT 
    c.CategoryName,
    COUNT(od.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY TotalOrders DESC
--Section 2:
SELECT 
    c.CategoryName,
    COUNT(od.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    SUM(od.LineAmount) / COUNT(od.OrderId) AS RevenuePerOrder
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY RevenuePerOrder ASC
--Section 3:
SELECT 
    c.CategoryName,
    COUNT(od.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    RANK() OVER (ORDER BY COUNT(od.OrderId) DESC) AS OrderRank,
    RANK() OVER (ORDER BY SUM(od.LineAmount) DESC) AS RevenueRank
FROM Production.Category c
JOIN Production.Product p ON c.CategoryId = p.CategoryId
JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
GROUP BY c.CategoryName
ORDER BY OrderRank ASC
--Section 4:
;WITH CategoryPerformance AS (
    SELECT 
        c.CategoryName,
        COUNT(od.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue,
        RANK() OVER (ORDER BY COUNT(od.OrderId) DESC) AS OrderRank,
        RANK() OVER (ORDER BY SUM(od.LineAmount) DESC) AS RevenueRank
    FROM Production.Category c
    JOIN Production.Product p ON c.CategoryId = p.CategoryId
    JOIN Sales.OrderDetail od ON p.ProductId = od.ProductId
    GROUP BY c.CategoryName
)
SELECT 
    CategoryName,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue,
    OrderRank,
    RevenueRank,
    OrderRank - RevenueRank AS RankDifference
FROM CategoryPerformance
ORDER BY RankDifference DESC


Explination: We want to find categories that look busy but undeliever on revenue. I did this by counting the orders, calculating the revenue per order, ranking both of those metrics and then used a CTE to compare those rankings. Again we see that Meat/Poultry has the biggest differnce of 4. It is ranked 7th in order volume but is 3rd in revenue. This means that it generates high value but it had low activity. Grains/Cereal had the lowest ranked difference being ranked 6th in orders but only 8th in revenue.

 Mystery 10:
Some customers keep coming back and spending big while other customers disappear.Find which customers are the most loyal and how much of our total revenue they are responsible for.

Section 1: Pull all customers and count their total orders

Section 2: Calculate total revenue per customer

Section 3: Calculate each customer's percentage share of total revenue

Section 4: Single CTE using window functions to rank customers by loyalty score combining order frequency and revenue

In [ ]:
--Section 1:
SELECT 
    c.CustomerId,
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalOrders DESC
--Section 2:
SELECT 
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY TotalRevenue DESC
--Section 3:
SELECT 
    c.CustomerCompanyName,
    COUNT(o.OrderId) AS TotalOrders,
    SUM(od.LineAmount) AS TotalRevenue,
    ROUND(SUM(od.LineAmount) * 100.0 / (SELECT SUM(LineAmount) FROM Sales.OrderDetail), 2) AS RevenueSharePct
FROM Sales.Customer c
JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
GROUP BY c.CustomerId, c.CustomerCompanyName
ORDER BY RevenueSharePct DESC
--Section 4:
;WITH CustomerLoyalty AS (
    SELECT 
        c.CustomerCompanyName,
        COUNT(o.OrderId) AS TotalOrders,
        SUM(od.LineAmount) AS TotalRevenue,
        ROUND(SUM(od.LineAmount) * 100.0 / (SELECT SUM(LineAmount) FROM Sales.OrderDetail), 2) AS RevenueSharePct
    FROM Sales.Customer c
    JOIN Sales.[Order] o ON c.CustomerId = o.CustomerId
    JOIN Sales.OrderDetail od ON o.OrderId = od.OrderId
    GROUP BY c.CustomerId, c.CustomerCompanyName
)
SELECT TOP 10
    CustomerCompanyName,
    TotalOrders,
    ROUND(TotalRevenue, 2) AS TotalRevenue,
    RevenueSharePct
FROM CustomerLoyalty
ORDER BY TotalRevenue DESC

Explination: We need to see who is our most loyal customer. Again we start by counting the orders but per customer, add the total revenue, calculate the revenue by share percentage and end it with a CTE that shows the top 10 customers. Customer IRRVL is our most loyal customer with generating $117,483.39 and being responsible for 8.67% of total revenue amongst 86 orders. Our top three customers are togther 25% of our revenue. We are heavily reliant on these 3 customers. 